In [ ]:
from jussam import load_data
from helpers import *

import numpy as np
from pandas import DataFrame

# 성능 평가 지표 (직접 계산하지 않고 sklearn 의 함수를 사용한다)
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error
from IPython.display import Markdown

c:\Users\itwill\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


📦 연세대학교 주영아 교수가 제작한 라이브러리를 사용중입니다.
📧 Email(1): j.purplerose@yonsei.ac.kr
📧 Email(2): j.purplerose@gmail.com
📝 Website: https://juyounga.kr/


In [3]:
df0 = load_data("insurance_checkpoint_0")
df1 = load_data("insurance_checkpoint_1")
df2 = load_data("insurance_checkpoint_2")
df3 = load_data("insurance_checkpoint_3")
df4 = load_data("insurance_checkpoint_4")

df0.head()

📚 보험 비용 데이터셋의 전처리 체크포인트 0 - 기준선 역할을 위해 라벨링과 더미변수 처리만 적용됨
📚 보험 비용 데이터셋의 전처리 체크포인트 1 - 라벨링 + 더미변수 + 로그변환
📚 보험 비용 데이터셋의 전처리 체크포인트 2 - 라벨링 + 더미변수 + 로그변환 + 이상치 대체
📚 보험 비용 데이터셋의 전처리 체크포인트 3 - 라벨링 + 더미변수 + 로그변환 + 이상치 대체 + 다중공선성제거
📚 보험 비용 데이터셋의 전처리 체크포인트 4 - 라벨링 + 더미변수 + 로그변환 + 이상치 대체 + 다중공선성제거 + 스케일링


,age,bmi,children,smoker,charges
0,19,27.900,0,1,16884.924
1,18,33.770,1,0,1725.552
2,28,33.000,3,0,4449.462
3,33,22.705,0,0,21984.471
4,32,28.880,0,0,3866.855


In [6]:
# 종속변수
target = "charges"

# 비교 대상 체크포인트 (아래로 갈수록 전처리가 하나씩 누적된다) // df0 ~ df4 까지 데이터 합친 것
checkpoints = { "0_기준선": df0, "1_로그변환": df1, "2_이상치대체": df2,
"3_공선성제거": df3, "4_스케일링": df4 }

# 종속변수(MEDV)에 log1p 가 적용되었는지 여부 (체크포인트 1부터 적용됨)
log1p_y = { "0_기준선": False, "1_로그변환": True, "2_이상치대체": True,
"3_공선성제거": True, "4_스케일링": True }

# 로그변환(log)이 적용된 독립변수 → 최종 모델의 회귀계수 해석에 사용한다
log_x = ["CRIM", "DIS", "RAD", "LSTAT"]

# 로그1p변환(log1p)이 적용된 독립변수 → 최종 모델의 회귀계수 해석에 사용한다
log1p_x = ["ZN"]

# 반전 로그 변환이 적용된 독립변수
reflect_x = ["B"]

### 종속변수 척도 확인

In [7]:
medv_scale = DataFrame()

for name in checkpoints: # 0_기준선, 1_로그변환... 등 딕셔너리의 key가 할당됨
    data = checkpoints[name] # df0 ~ df4 까지 복사
    medv_scale[name] = data[target].describe() # 복사된 데이터에서 종속변수의 기술통계량만 구함

medv_scale.round(3)

,0_기준선,1_로그변환,2_이상치대체,3_공선성제거,4_스케일링
count,1337.000,1337.000,1337.000,1337.000,1337.000
mean,13279.121,9.100,9.100,9.100,9.100
std,12110.360,0.919,0.919,0.919,0.919
min,1121.874,7.023,7.023,7.023,7.023
25%,4746.344,8.465,8.465,8.465,8.465
50%,9386.161,9.147,9.147,9.147,9.147
75%,16657.717,9.721,9.721,9.721,9.721
max,63770.428,11.063,11.063,11.063,11.063


### 체크포인트마다 모델을 적합 

In [8]:
fits = {}   # 체크포인트별 학습 객체를 저장할 딕셔너리
result = [] # 체크포인트별 학습 객체를 통해 모델의 성능 평가 지표를 저장할 리스트 (데이터프레임 변환 목적)

for c in checkpoints:
    data = checkpoints[c]   # 체크포인트 데이터프레임
    log = log1p_y[c]        # 종속변수에 log1p 가 적용되었는지 여부 (T/F)

    # --- 1) 모델 적합 (후진제거 포함) ---
    fit = my_ols.auto_ols(data, target, log1p_y=log, log_x=log_x, log1p_x=log1p_x, reflect_x=reflect_x,
                          backward=True, report=False, test=False)

    fits[c] = fit # 체크포인트별 학습 객체를 딕셔너리 형태로 저장한다

    # --- 2) 실제값과 예측값을 원본 척도 (천 달러)로 되돌린다 ---
    if log:
        y_true = np.expm1(data[target])     # 종속변수 역변환
        y_pred = np.expm1(fit.fittedvalues) # 예측값 역변환
    else:
        y_true = data[target]               # 체크포인트 0 은 이미 원본 척도
        y_pred = fit.fittedvalues

    # --- 3) 지표를 한 행으로 정리해 저장 ---
    result.append({
        "모델": c,
        "변수수": int(fit.df_model),    # 상수항을 제외한 독립변수 개수
        "R2(모델척도)": fit.rsquared,   # 체크포인트간 비교 불가 (참고용)
        "Adj.R2": fit.rsquared_adj,    # 같은 종속변수끼리만 비교
        "AIC": fit.aic,                # 같은 종속변수끼리만 비교 (작을수록 좋음)
        "BIC": fit.bic,                # 같은 종속변수끼리만 비교 (작을수록 좋음)
        # 아래 세 지표는 원본 척도에서 계산하므로 체크포인트간 비교가 가능하다
        "R2(원본척도)": r2_score(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred), # 최종 선택 기준
        "MAE": mean_absolute_error(y_true, y_pred),
        })

rdf = DataFrame(result).set_index('모델')
rdf.round(4)

,변수수,R2(모델척도),Adj.R2,AIC,BIC,R2(원본척도),RMSE,MAE
모델,,,,,,,,
0_기준선,4,0.750,0.749,27092.752,27118.743,0.750,6058.641,4181.335
1_로그변환,4,0.763,0.762,1653.311,1679.302,0.532,8281.267,4247.790
2_이상치대체,4,0.763,0.762,1652.808,1678.799,0.532,8278.466,4248.574
3_공선성제거,4,0.763,0.762,1652.808,1678.799,0.532,8278.466,4248.574
4_스케일링,4,0.763,0.762,1652.808,1678.799,0.532,8278.466,4248.574


### 2. 최종 모델 채택

In [12]:
final_fit = fits['0_기준선']
final_data = checkpoints['0_기준선']

In [13]:
display(Markdown(
    my_ols.report_fitness(final_fit, log1p_y=True, log_x=log_x,
                          log1p_x=log1p_x, reflect_x=reflect_x)
                          ))

**Note. n = 1337. F(4, 1332) = 996.48, p < 0.001, R² = 0.75, Adj.R² = 0.749, Durbin-Watson = 2.088**

log1p(charges)를 종속변수로, age, bmi, children, smoker(을)를 독립변수로 한 다중선형회귀분석 결과, 모형은 통계적으로 유의하였다.

> F(4, 1332) = 996.48, p < 0.001, R² = 0.75.

즉, age, bmi, children, smoker는 log1p(charges)의 약 74.95%를 설명하는 것으로 나타났다.